In [1]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [03:25<00:00, 833.02it/s]


In [4]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:17<00:00, 9630.14it/s]


In [5]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:07<00:00, 21877.66it/s]


In [6]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.75, b=0.50):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [7]:
query = 'what is the origin of COVID-19'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 2048/2048 [00:00<00:00, 223132.57it/s]


[('vh96sjss', 6.8606865201471265),
 ('dv9m19yk', 6.834467053049621),
 ('8ccl9aui', 6.563258551880041),
 ('4uaa6kpg', 6.4750694087288),
 ('icwvm7jp', 6.424394377576402),
 ('deajwhx0', 6.424394377576402),
 ('wim5q9a5', 6.403963857839434),
 ('d0x23frk', 6.396582832347495),
 ('ymhcouo5', 6.374506370435119),
 ('49360l2a', 6.336237036056754),
 ('9l97eihy', 6.282356871665615),
 ('v6ci69n0', 6.226487251451357),
 ('us1spoxu', 6.2241372548436456),
 ('73ylxhb7', 6.219498298986689),
 ('42wv7zl6', 6.210174436645219),
 ('e3wjo0yk', 6.179076742677619),
 ('2qto9vsb', 6.100783907624839),
 ('z14rf85c', 6.089626973029883),
 ('a8h7irel', 6.083358514167039),
 ('2ntxpdke', 6.043136415092819),
 ('4dtk1kyh', 6.010361120885761),
 ('l0kc731z', 5.97904884290847),
 ('lc7hkgka', 5.9677282521679595),
 ('d1p9e5sm', 5.9650684772600355),
 ('cniyembt', 5.959756033768593),
 ('kw7mon2o', 5.952252833410947),
 ('zd7smm8r', 5.950156243586571),
 ('ayj4z8qn', 5.931049143559384),
 ('ceehbhcb', 5.914134190973142),
 ('pivqu9bu',